In [0]:
%pip install -U faster-whisper mlflow databricks-sdk

In [0]:
%restart_python

In [0]:
import os
import nvidia.cublas.lib
import nvidia.cudnn.lib
print(os.path.dirname(nvidia.cublas.lib.__file__) + ":" + os.path.dirname(nvidia.cudnn.lib.__file__))

In [0]:
from faster_whisper import WhisperModel
import torch

In [0]:
device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = "bfloat16"
model_size = "large-v3"

model = WhisperModel(model_size, device=device, compute_type=torch_dtype)

In [0]:
data_path = "/Volumes/raghu_demos_uc/whisper_models/audio_files/1/DATASET/Environment/crowd/crowd_1_part_1.wav"

segments, info = model.transcribe(data_path, beam_size=5)

print("Detected language '%s' with probability %f" % (info.language, info.language_probability))

for segment in segments:
    print("[%.2fs -> %.2fs] %s" % (segment.start, segment.end, segment.text))

In [0]:
from mlflow.models.signature import infer_signature, ModelSignature
import requests
import io
import base64
import pandas as pd

# Load an audio file (Replace with a valid file path)
data_path = "/Volumes/raghu_demos_uc/whisper_models/audio_files/1/DATASET/Environment/crowd/crowd_1_part_1.wav"
with open(data_path, "rb") as f:
    audio_bytes = f.read()  # Read MP3 as raw bytes
    # audio_string = base64.b64encode(audio_bytes).decode("utf-8")

audio_df = pd.DataFrame({"audio": [audio_bytes]})

params = {"temperature": 0.1, "beam_size":5}
output_example = pd.DataFrame().from_records([{"output_text": "this is an example output"}])

signature = infer_signature(audio_df, output_example, params)
print(signature)

In [0]:
from mlflow.pyfunc import PythonModel
from io import BytesIO
import os
import logging
import subprocess
import os
import nvidia.cublas.lib
import nvidia.cudnn.lib

class FasterWhisperModel(PythonModel):

    def __init__(self):
        self.logger = logging.getLogger(__name__)
        self.logger.setLevel(logging.WARNING)

    def load_context(self, context):
      self.device = "cuda" if torch.cuda.is_available() else "cpu"
      self.torch_dtype = "bfloat16"
      self.model_size = os.getenv("WHISPER_MODEL_SIZE", default="large-v3")
      self.model = WhisperModel(self.model_size, 
                                device=self.device, 
                                compute_type=self.torch_dtype)

    def predict(self, context, model_input, params):
      if not params:
        params=dict()
      temperature = params.get("temperature", 0.1)
      beam_size = params.get("beam_size", 5)

      transcriptions = []
      for audio_bytes in model_input["audio"]:
          segments, _ = self.model.transcribe(audio=BytesIO(audio_bytes),
                                              temperature=temperature,
                                              beam_size=beam_size)
          transcript = " ".join([segment.text for segment in segments])
          transcriptions.append(transcript)
      return pd.DataFrame({"output_text": transcriptions})


In [0]:
import mlflow

with mlflow.start_run():
    logged_model_info = mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=FasterWhisperModel(),
        input_example=audio_df,
        signature=signature,
    )

In [0]:
loaded_model = mlflow.pyfunc.load_model("runs:/"+logged_model_info.run_id+"/model")
output = loaded_model.predict(audio_df)
display(output)

In [0]:
mlflow.set_registry_uri("databricks-uc")

mlflow.register_model("runs:/"+logged_model_info.run_id+"/model", "uc_sriharsha_jana.test_db.faster_whisper")

In [0]:
%sh
nvidia-smi

In [0]:
# Define the model serving endpoint configuration
import os
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput, ServingModelWorkloadType

In [0]:
# Create a WorkspaceClient instance
db_client = WorkspaceClient()

In [0]:
# Create serving endpoint configuration
UC_MODE_NAME = "uc_sriharsha_jana.test_db.faster_whisper"
ENDPOINT_NAME = "shj-faster-whisper"

endpoint_config = EndpointCoreConfigInput(
    served_entities=[
        ServedEntityInput(
            entity_name=UC_MODE_NAME,  # Full Unity Catalog path
            entity_version="8",                       # Model version
            workload_size="Small",                    # Compute size
            scale_to_zero_enabled=True,
            workload_type=ServingModelWorkloadType.GPU_MEDIUM,
            environment_vars={
                "WHISPER_MODEL_SIZE": "large-v3",
                "LD_LIBRARY_PATH":"/opt/conda/envs/mlflow-env/lib/python3.12/site-packages/nvidia/cudnn/lib:/opt/conda/envs/mlflow-env/lib/python3.12/site-packages/nvidia/cublas/lib"
                }
        )
    ]
)

# Create and deploy the endpoint
endpoint = db_client.serving_endpoints.create(
    name=ENDPOINT_NAME,
    config=endpoint_config
).result()

# Wait for deployment completion
db_client.serving_endpoints.wait_get_serving_endpoint_not_updating(
    name=ENDPOINT_NAME
)

print(f"Endpoint created: {endpoint.name} (State: {endpoint.state.ready})")